# CarMax ABS — loan-level visibility

**Where the data lives.** The `srs` monorepo has no CarMax ABS pipeline. `workflows/jobs/carmax/` only builds retail alt-data (M Science + YipitData → `MSCIENCE.KMX.{KPIS,SIGNALS}`). The ABS data sits in Snowflake under `BOTS`, and no code in the monorepo writes to it:

| Table | Rows | Notes |
|---|---|---|
| `BOTS.AUTOABS.CARMAX_DETAIL` | ~38.3M | Raw ABS-EE (EX-102) loan-level, CarMax Auto Owner Trusts, periods 2021-09 → 2025-12 |
| `BOTS.AUTOABS.CARMAX_SEC2_DETAIL` | ~8.3M | Second scrape, periods 2025-08 → 2026-01 (latest) |
| `BOTS.AUTOABS.CARMAX_SEC2_LIST` | 116 | Filing index (trust, filed date, ABS-EE link) |
| `BOTS.KMX.ABS_COMBINED_PROCESSED` | ~24.1M | Multi-issuer combined + derived fields (`LOAN_EXTENDED`, `PREV_*`, `SUDDEN_PAID_OFF_WHILE_DELINQUENT60DAYS`) |

⚠️ The last load was about Feb 2026, so the feed looks stale as of Oct 2026. Confirm with the `BOTS` owner whether the scraper is still running.

In [2]:
import os
import pandas as pd
import snowflake.connector

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

# Uses a named connection from ~/.snowflake/connections.toml (same pattern as srs-evals).
conn = snowflake.connector.connect(
    connection_name=os.getenv("SNOWFLAKE_CONNECTION", "default"),
)

def q(sql: str) -> pd.DataFrame:
    df = conn.cursor().execute(sql).fetch_pandas_all()
    df.columns = [c.lower() for c in df.columns]
    return df

RAW = "BOTS.AUTOABS.CARMAX_DETAIL"
SEC2 = "BOTS.AUTOABS.CARMAX_SEC2_DETAIL"
PROC = "BOTS.KMX.ABS_COMBINED_PROCESSED"

 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...


DatabaseError: 251016 (08001): Failed to connect to DB: PA12850-XT39849.snowflakecomputing.com:443, Unable to receive the OAuth message within a given timeout. Please check the redirect URI and try again.

## 1. Coverage and freshness

In [ ]:
q(f"""
SELECT '{RAW}' AS src, COUNT(*) AS n_rows, COUNT(DISTINCT trust_name) AS trusts,
       MIN(reportingperiodendingdate) AS min_period, MAX(reportingperiodendingdate) AS max_period,
       MAX(filing_date) AS max_filing, MAX(insertupdatetime)::STRING AS last_load
FROM {RAW}
UNION ALL
SELECT '{SEC2}', COUNT(*), COUNT(DISTINCT trust_name), MIN(reportingperiodendingdate), MAX(reportingperiodendingdate),
       MAX(filing_date), MAX(insertupdatetime)::STRING
FROM {SEC2}
UNION ALL
SELECT '{PROC} (CarMax only)', COUNT(*), COUNT(DISTINCT trust_name), MIN(reportingperiodendingdate), MAX(reportingperiodendingdate),
       MAX(filing_date), MAX(insertupdatetime)::STRING
FROM {PROC} WHERE trust_name ILIKE '%carmax%'
""")

In [ ]:
# Trust x period grid: loan counts per monthly tape
trusts = q(f"""
SELECT trust_name, reportingperiodendingdate AS period, COUNT(*) AS loans,
       SUM(reportingperiodactualendbalanceamount) / 1e6 AS end_bal_mm
FROM {SEC2}
GROUP BY 1, 2
""")
trusts.pivot(index="trust_name", columns="period", values="loans")

## 2. Loan-level field fill rates (latest period)

This is the main visibility check: which EX-102 fields are actually populated.

In [ ]:
def fill_rates(table: str) -> pd.DataFrame:
    db, schema, name = table.split(".")
    cols = q(f"""
        SELECT column_name, data_type FROM {db}.INFORMATION_SCHEMA.COLUMNS
        WHERE table_schema = '{schema}' AND table_name = '{name}' ORDER BY ordinal_position
    """)
    exprs = ",\n".join(
        f"COUNT({c}) / COUNT(*) AS {c}, COUNT(DISTINCT {c}) AS {c}__distinct" for c in cols.column_name
    )
    row = q(f"""
        SELECT {exprs} FROM {table}
        WHERE reportingperiodendingdate = (SELECT MAX(reportingperiodendingdate) FROM {table})
    """).iloc[0]
    out = cols.assign(
        fill_rate=[row[c.lower()] for c in cols.column_name],
        n_distinct=[row[f"{c.lower()}__distinct"] for c in cols.column_name],
    )
    return out.set_index("column_name")

fr = fill_rates(SEC2)
fr.style.format({"fill_rate": "{:.1%}"}).background_gradient(subset=["fill_rate"], cmap="RdYlGn")

## 3. Sample loans

In [ ]:
q(f"""
SELECT * FROM {SEC2}
WHERE reportingperiodendingdate = (SELECT MAX(reportingperiodendingdate) FROM {SEC2})
SAMPLE (10 ROWS)
""").T

## 4. Follow a single loan through time

`ASSETNUMBER` should be stable within a trust. This confirms we can build loan-level performance histories (roll rates, cures, extensions, charge-offs).

In [ ]:
# Pick a loan that has gone delinquent at some point in the raw history
pick = q(f"""
SELECT trust_name, assetnumber FROM {RAW}
WHERE currentdelinquencystatus >= 30
SAMPLE (1 ROWS)
""").iloc[0]

q(f"""
SELECT reportingperiodendingdate, reportingperiodbeginningloanbalanceamount, reportingperiodactualendbalanceamount,
       reportingperiodscheduledpaymentamount, totalactualamountpaid, currentdelinquencystatus,
       paymentextendednumber, modificationtypecode, loanmaturitydate, zerobalancecode,
       chargedoffprincipalamount, recoveredamount, repossessedindicator
FROM {RAW}
WHERE trust_name = '{pick.trust_name}' AND assetnumber = '{pick.assetnumber}'
ORDER BY reportingperiodendingdate
""")

In [ ]:
# Is (trust, assetnumber, period) unique? Duplicates would indicate re-filings or overlapping scrapes.
for t in (RAW, SEC2):
    display(q(f"""
    SELECT '{t}' AS src, COUNT(*) AS rows_, COUNT(DISTINCT trust_name, assetnumber, reportingperiodendingdate) AS uniq_keys
    FROM {t}
    """))

## 5. Origination profile by trust (credit box)

In [ ]:
orig = q(f"""
WITH first_obs AS (
    SELECT * FROM {RAW}
    QUALIFY ROW_NUMBER() OVER (PARTITION BY trust_name, assetnumber ORDER BY reportingperiodendingdate) = 1
)
SELECT trust_name,
       COUNT(*)                                                    AS loans,
       AVG(originalloanamount)                                     AS avg_amt,
       AVG(originalloanterm)                                       AS avg_term,
       AVG(originalinterestratepercentage) * 100                   AS avg_apr_pct,
       AVG(obligorcreditscore)                                     AS avg_fico,
       MEDIAN(obligorcreditscore)                                  AS med_fico,
       AVG(originalloanamount / NULLIF(vehiclevalueamount, 0))     AS avg_ltv,
       AVG(paymenttoincomepercentage) * 100                        AS avg_pti_pct,
       AVG(IFF(coobligorindicator, 1, 0))                          AS pct_coobligor,
       AVG(vehiclemodelyear)                                       AS avg_model_year
FROM first_obs
GROUP BY 1 ORDER BY 1
""")
orig

## 6. Pool performance from loan-level data

`CURRENTDELINQUENCYSTATUS` is days past due.

In [ ]:
perf = q(f"""
SELECT reportingperiodendingdate AS period,
       SUM(reportingperiodactualendbalanceamount)                                                      AS end_bal,
       SUM(IFF(currentdelinquencystatus >= 30, reportingperiodactualendbalanceamount, 0)) / NULLIF(end_bal, 0) AS dq30_pct,
       SUM(IFF(currentdelinquencystatus >= 60, reportingperiodactualendbalanceamount, 0)) / NULLIF(end_bal, 0) AS dq60_pct,
       SUM(chargedoffprincipalamount)                                                                   AS chargeoffs,
       SUM(recoveredamount)                                                                             AS recoveries,
       SUM(IFF(paymentextendednumber > 0, 1, 0))                                                        AS loans_extended
FROM {RAW}
GROUP BY 1 ORDER BY 1
""")
perf.set_index("period")[["dq30_pct", "dq60_pct"]].plot(title="CarMax ABS — balance-weighted delinquency (all trusts)", figsize=(10, 4));

## 7. Processed table: derived fields

Check what `BOTS.KMX.ABS_COMBINED_PROCESSED` adds on top of the raw tape, and which issuers it covers.

In [ ]:
q(f"""
SELECT SPLIT_PART(trust_name, ' ', 1) AS issuer, COUNT(*) AS n_rows,
       MIN(reportingperiodendingdate) AS min_period, MAX(reportingperiodendingdate) AS max_period,
       AVG(loan_extended) AS pct_extended,
       AVG(sudden_paid_off_while_delinquent60days) AS pct_sudden_payoff_dq60
FROM {PROC}
GROUP BY 1 ORDER BY 2 DESC
""")